In [124]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [125]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [126]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [127]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [128]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [129]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [130]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [131]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

- model

In [132]:
class MyNN():
    def __init__(self,X):
        self.weights = torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
        self.bias = torch.zeros(1,dtype=torch.float64,requires_grad=True)

    def forward(self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_fuction(self,y_pred,y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)

        loss = -(y_train_tensor * torch.log(y_pred) + (1- y_train_tensor)*torch.log(1-y_pred)).mean()
        return loss

In [133]:
learning_rate = 0.1
epochs =25

In [134]:
# create model
model = MyNN(X_train_tensor)

for i in range(epochs):
        # forward pass
        y_pred =  model.forward(X_train_tensor)

        # Loss 
        loss = model.loss_fuction(y_pred,y_train_tensor)

        # backward
        loss.backward()

        # update parameters
        with torch.no_grad():
                model.weights -= learning_rate * model.weights.grad 
                model.bias -= learning_rate * model.bias.grad 

        #zero Gradient
        model.weights.grad.zero_()
        model.bias.grad.zero_()

        # loss in each epochs
        print(f'Epoch: {i+1},loss: {loss.item()}')


Epoch: 1,loss: 2.995292335113368
Epoch: 2,loss: 2.8536414322524513
Epoch: 3,loss: 2.7102471508742485
Epoch: 4,loss: 2.5626876197749993
Epoch: 5,loss: 2.41817337124783
Epoch: 6,loss: 2.2738222689327667
Epoch: 7,loss: 2.1250029480839263
Epoch: 8,loss: 1.9746854301732988
Epoch: 9,loss: 1.829635438806806
Epoch: 10,loss: 1.6884252478806694
Epoch: 11,loss: 1.550833084537149
Epoch: 12,loss: 1.4244408352438849
Epoch: 13,loss: 1.310255411820515
Epoch: 14,loss: 1.2091705569616007
Epoch: 15,loss: 1.1218221160283373
Epoch: 16,loss: 1.0483467268282887
Epoch: 17,loss: 0.9881493464719615
Epoch: 18,loss: 0.9398709099925802
Epoch: 19,loss: 0.9016272346327847
Epoch: 20,loss: 0.8713661684132089
Epoch: 21,loss: 0.8471757711326209
Epoch: 22,loss: 0.8274803580057686
Epoch: 23,loss: 0.8111001455823296
Epoch: 24,loss: 0.7972015122056342
Epoch: 25,loss: 0.7852065390387232


- evalution

In [135]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(accuracy)


tensor(0.5462)


## through nn module

In [136]:
import torch.nn as nn

In [137]:
class mynnmodule(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out

    

In [138]:
loss_function = nn.BCELoss()
type(loss_function)

torch.nn.modules.loss.BCELoss

In [139]:
# create model
model = mynnmodule(X_train_tensor.shape[1])
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()
X_test_tensor = X_test_tensor.float()
y_test_tensor = y_test_tensor.float()

optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)
for i in range(epochs):
        # forward pass
        y_pred =  model(X_train_tensor)

        # Loss 
        loss = loss_function(y_pred,y_train_tensor.reshape(-1,1))

        # backward
        optimizer.zero_grad()
        loss.backward()

        # update parameters
        # with torch.no_grad():
        #         model.linear.weight -= learning_rate * model.linear.weight.grad 
        #         model.linear.bias -= learning_rate * model.linear.bias.grad 



        optimizer.step()

        # #zero Gradient
        # model.linear.weight.grad.zero_()
        # model.linear.bias.grad.zero_()
        

        # loss in each epochs
        print(f'Epoch: {i+1},loss: {loss.item()}')


Epoch: 1,loss: 0.7372623682022095
Epoch: 2,loss: 0.563306987285614
Epoch: 3,loss: 0.46598389744758606
Epoch: 4,loss: 0.40601950883865356
Epoch: 5,loss: 0.36498305201530457
Epoch: 6,loss: 0.33482927083969116
Epoch: 7,loss: 0.3115491271018982
Epoch: 8,loss: 0.29291394352912903
Epoch: 9,loss: 0.2775813043117523
Epoch: 10,loss: 0.2646915018558502
Epoch: 11,loss: 0.2536667585372925
Epoch: 12,loss: 0.24410314857959747
Epoch: 13,loss: 0.235709086060524
Epoch: 14,loss: 0.22826819121837616
Epoch: 15,loss: 0.2216162085533142
Epoch: 16,loss: 0.2156258076429367
Epoch: 17,loss: 0.21019667387008667
Epoch: 18,loss: 0.20524851977825165
Epoch: 19,loss: 0.20071610808372498
Epoch: 20,loss: 0.19654595851898193
Epoch: 21,loss: 0.19269360601902008
Epoch: 22,loss: 0.18912184238433838
Epoch: 23,loss: 0.18579918146133423
Epoch: 24,loss: 0.18269884586334229
Epoch: 25,loss: 0.17979779839515686


In [142]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred.squeeze() > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(accuracy)


tensor(0.9561)
